In [ ]:


import pandas as pd
import numpy as np

# ------------------------------------------------
# STEP 1 — Load datasets
# ------------------------------------------------

print("Loading datasets...")

df = pd.read_csv("HAZARD_EVENT_WINDOWS.csv")
clusters = pd.read_csv("HAZARD_CLUSTER_TABLE.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["hazard_time"] = pd.to_datetime(df["hazard_time"], utc=True)

# Create t_offset (seconds from hazard moment)
df["t_offset"] = (df["timestamp"] - df["hazard_time"]).dt.total_seconds()


# ------------------------------------------------
# STEP 2 — Recalculate missing feature columns
# ------------------------------------------------

print("Computing gps_speed...")

# Shift coordinates for distance calculation
df["lat_shift"] = df.groupby("hazard_id")["lat"].shift(1)
df["lon_shift"] = df.groupby("hazard_id")["lon"].shift(1)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df["gps_speed"] = haversine(
    df["lat"], df["lon"],
    df["lat_shift"], df["lon_shift"]
)


print("Computing rolling acceleration features...")

# Rolling forward acceleration statistics
df["accx_5s_mean"] = df.groupby("hazard_id")["accx_g"].rolling(5, min_periods=1).mean().values
df["accx_5s_min"]  = df.groupby("hazard_id")["accx_g"].rolling(5, min_periods=1).min().values


print("Computing black ice risk...")

df["black_ice_risk"] = (
    (df["surface_temperature"] < 2) &
    ((df["air_temperature"] - df["surface_temperature"]) > 0)
).astype(int)


# ------------------------------------------------
# STEP 3 — Extract pre-hazard window (−5s to −1s)
# ------------------------------------------------

print("Extracting -5s to -1s pre-hazard window...")

pre_df = df[(df["t_offset"] >= -5) & (df["t_offset"] < 0)].copy()
print("Pre-hazard rows:", len(pre_df))


# ------------------------------------------------
# STEP 4 — Aggregate features per hazard
# ------------------------------------------------

print("Aggregating pre-hazard features...")

pre_features = pre_df.groupby("hazard_id").agg({
    "gps_speed": ["mean", "min"],
    "accx_5s_mean": "mean",
    "accx_5s_min": "min",
    "grip": "mean",
    "water_layer_thickness": "mean",
    "ice_layer_thickness": "mean",
    "snow_layer_thickness": "mean",
    "black_ice_risk": "mean"
})

# Flatten multi-index columns
pre_features.columns = ['_'.join(col) for col in pre_features.columns]
pre_features.reset_index(inplace=True)


# ------------------------------------------------
# STEP 5 — Merge K-means cluster labels
# ------------------------------------------------

print("Merging cluster labels...")

pre_features = pre_features.merge(
    clusters[["hazard_id", "cluster"]],
    on="hazard_id",
    how="left"
)


# ------------------------------------------------
# STEP 6 — Prepare data for ML (Fix NaN, extract X & y)
# ------------------------------------------------

print("Preparing ML dataset...")

pre_features = pre_features.fillna(pre_features.median())

feature_cols = [c for c in pre_features.columns if c not in ["hazard_id", "cluster"]]

X = pre_features[feature_cols]
y = pre_features["cluster"]


# ------------------------------------------------
# STEP 7 — Train/Test split
# ------------------------------------------------

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)


# ------------------------------------------------
# STEP 8 — Random Forest (Pre-hazard prediction)
# ------------------------------------------------

print("Training Random Forest...")

from sklearn.ensemble import RandomForestClassifier

rf_pre = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42
)

rf_pre.fit(X_train, y_train)


# RF Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_pred = rf_pre.predict(X_test)

print("\n============================")
print(" RANDOM FOREST RESULTS")
print("============================")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))


# ------------------------------------------------
# STEP 9 — XGBoost (Best model for prediction)
# ------------------------------------------------

print("Training XGBoost...")

from xgboost import XGBClassifier

xgb_pre = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softmax",
    num_class=len(y.unique()),
    eval_metric="mlogloss",
    random_state=42
)

xgb_pre.fit(X_train, y_train)

xgb_pred = xgb_pre.predict(X_test)


print("\n============================")
print(" XGBOOST RESULTS")
print("============================")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, xgb_pred))
print("\nClassification Report:\n", classification_report(y_test, xgb_pred))


print("\n\nALL DONE — PRE-HAZARD PREDICTION COMPLETE!")